In [145]:
import pandas as pd

In [146]:
df=pd.read_csv('auto-mpg.csv')
df

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino
...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86,2790,15.6,82,1,ford mustang gl
394,44.0,4,97.0,52,2130,24.6,82,2,vw pickup
395,32.0,4,135.0,84,2295,11.6,82,1,dodge rampage
396,28.0,4,120.0,79,2625,18.6,82,1,ford ranger


In [147]:
df['horsepower']=pd.to_numeric(df['horsepower'],errors='coerce')

In [148]:
df=df.drop(columns=['car name','origin'])

In [149]:
target_variable_name='mpg'
X=df.drop(columns=[target_variable_name])
y=df[target_variable_name]

In [150]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [151]:
X_num_col_names=X.select_dtypes(include=['number']).columns
X_cat_col_names=X.select_dtypes(exclude=['number']).columns

In [152]:
# numerical features outliers handling

q1=X_train[X_num_col_names].quantile(0.25)
q3=X_train[X_num_col_names].quantile(0.75)
iqr=q3-q1
lower_fence=q1-1.5*iqr
upper_fence=q3+1.5*iqr

mask=((X_train[X_num_col_names]<lower_fence)|(X_train[X_num_col_names]>upper_fence)).any(axis=1)
X_train=X_train[~mask]
y_train=y_train[~mask]

In [153]:
from sklearn.impute import KNNImputer

# removal of rows with missing values at target
mask_y_train=~y_train.isnull()
mask_y_test=~y_test.isnull()
y_train=y_train[mask_y_train]
y_test=y_test[mask_y_test]
X_train=X_train[mask_y_train]
X_test=X_test[mask_y_test]

# removal of numeric features with high missing value ratio and removal of categorical features with moderate-high missing value ratio
num_mask_remove_features=(X_train[X_num_col_names].isnull().mean()>=0.5).reindex(X_train.columns,fill_value=False)
cat_mask_remove_features=(X_train[X_cat_col_names].isnull().mean()>=0.2).reindex(X_train.columns,fill_value=False)
mask_remove_features=num_mask_remove_features|cat_mask_remove_features
X_train=X_train.drop(columns=mask_remove_features[mask_remove_features].index)
X_test=X_test.drop(columns=mask_remove_features[mask_remove_features].index)
X_num_col_names=X_num_col_names.intersection(X_train.columns)
X_cat_col_names=X_cat_col_names.intersection(X_train.columns)

# imputation(numeric) of rows in low-moderate missing value ratio features
# (using knn imputer that uses train features and work on train/validation/test features)
if len(X_num_col_names)>0:
    knn_imputer=KNNImputer(n_neighbors=5)
    X_train[X_num_col_names]=knn_imputer.fit_transform(X_train[X_num_col_names])
    X_test[X_num_col_names]=knn_imputer.transform(X_test[X_num_col_names])

# imputation(categorical) of rows in low missing value ratio features
# (using mode imputation using mode of train features that is also used in imputation of train/validation/test features)
if len(X_cat_col_names)>0:
    modes=X_train[X_cat_col_names].mode().iloc[0]
    X_train[X_cat_col_names]=X_train[X_cat_col_names].fillna(modes)
    X_test[X_cat_col_names]=X_test[X_cat_col_names].fillna(modes)

In [154]:
from sklearn.preprocessing import StandardScaler
standard_scaler=StandardScaler()
X_train[X_num_col_names]=standard_scaler.fit_transform(X_train[X_num_col_names])
X_test[X_num_col_names]=standard_scaler.transform(X_test[X_num_col_names])

In [155]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
rmse=root_mean_squared_error(y_test,y_pred)
print("RMSE:",rmse)

RMSE: 3.342993037192852
